# Customer Catalogue GP Homologation

## Step 1 - Load, validate and normalize Global Parents

This notebook:

- Loads the customer catalogue
- Validates required columns
- Counts unique Ship To Numbers per Global Parent
- Creates a normalized GP name
- Exports a summary for review

The original catalogue is NEVER modified.

In [ ]:
import pandas as pd
import re
import unicodedata
from pathlib import Path

In [ ]:
# ==========================================
# Configuration
# ==========================================

INPUT_FILE = "CustomerCatalogue.csv"

OUTPUT_FOLDER = "output"

GP_COLUMN = "Global Parent"

SHIP_TO_COLUMN = "Ship To Number"

REMOVE_ACCENTS = True
REMOVE_PUNCTUATION = True
REMOVE_EXTRA_SPACES = True
REMOVE_LEGAL_SUFFIXES = True

In [ ]:
LEGAL_SUFFIXES = [

    "INCORPORATED",
    "INC",
    "CORPORATION",
    "CORP",
    "COMPANY",
    "CO",
    "LIMITED",
    "LTD",
    "LLC",
    "LLP",
    "PLC",
    "LP",
    "GMBH",
    "AG",
    "BV",
    "NV",
    "SA",
    "SAS",
    "SPA",
    "SRL",
    "PTY",
    "PVT",
    "KK"

]

In [ ]:
def normalize_gp(name):

    if pd.isna(name):
        return ""

    text = str(name).upper()

    if REMOVE_ACCENTS:
        text = unicodedata.normalize("NFKD", text)
        text = "".join(c for c in text if not unicodedata.combining(c))

    if REMOVE_PUNCTUATION:
        text = re.sub(r"[^\w\s]", " ", text)

    if REMOVE_LEGAL_SUFFIXES:

        words = text.split()

        words = [
            word
            for word in words
            if word not in LEGAL_SUFFIXES
        ]

        text = " ".join(words)

    if REMOVE_EXTRA_SPACES:
        text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df):,}")

In [ ]:
required_columns = [

    GP_COLUMN,
    SHIP_TO_COLUMN

]

missing = [

    column
    for column in required_columns
    if column not in df.columns

]

if missing:

    raise ValueError(
        f"Missing columns: {missing}"
    )

print("All required columns found.")

In [ ]:
df["Normalized GP"] = df[GP_COLUMN].apply(normalize_gp)

In [ ]:
gp_summary = (

    df.groupby(GP_COLUMN)
      .agg(

          Ship_To_Count=(
              SHIP_TO_COLUMN,
              "nunique"
          ),

          Row_Count=(
              SHIP_TO_COLUMN,
              "count"
          ),

          GP_Country=(
              "GP Country",
              "first"
          ),

          Normalized_GP=(
              "Normalized GP",
              "first"
          )

      )

      .reset_index()

)

In [ ]:
gp_summary = gp_summary.sort_values(

    by=[
        "Ship_To_Count",
        GP_COLUMN
    ],

    ascending=[
        False,
        True
    ]

)

In [ ]:
gp_summary.head(20)

In [ ]:
Path(OUTPUT_FOLDER).mkdir(exist_ok=True)

output_file = Path(OUTPUT_FOLDER) / "01_gp_summary.csv"

gp_summary.to_csv(

    output_file,

    index=False

)

print(f"Saved to: {output_file}")